[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Saving and Loading Models &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os
import shutil
import zipfile
from pathlib import Path

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


def station_network_arrays(n=1200, seed=2025):
    """The made-up stations of the scikit-learn guide, as arrays: four features and two targets."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    shift = np.where(coast_km < 3, 0.8,
            np.where(elevation > 700, -0.5,
            np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), 0.3, 0.0)))
    rng.choice(4, n, p=[0.4, 0.3, 0.1, 0.2])   # each station's owner, which this guide leaves out
    rng.integers(1950, 2025, n)                 # and the year it was installed
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    features = np.column_stack([latitude.round(2), longitude.round(2), elevation.round(0), coast_km.round(1)])
    return features.astype("float32"), jan_mean.round(1).astype("float32"), (may_mean < 2.5).astype("float32")


FEATURES = ["latitude", "longitude", "elevation_m", "coast_km"]
features, jan_mean, snow_on_may_1 = station_network_arrays()
order = np.random.default_rng(0).permutation(len(features))
train_rows, valid_rows, test_rows = order[:800], order[800:1000], order[1000:]
mean, std = features[train_rows].mean(axis=0), features[train_rows].std(axis=0)
scaled = (features - mean) / std

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(features), "stations")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 1200 stations


The worked examples' network and training loop, and `net` and `net_keras` trained again on the same
made-up stations from the same seeds, so that each answer runs on its own.

In [2]:
X_train = torch.from_numpy(scaled[train_rows])
y_train = torch.from_numpy(jan_mean[train_rows]).unsqueeze(1)
loss_fn = torch.nn.MSELoss()


def make_net():
    return torch.nn.Sequential(torch.nn.Linear(4, 16), torch.nn.ReLU(), torch.nn.Linear(16, 1))


def train(model, optimizer, steps):
    for _ in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model(X_train), y_train)
        loss.backward()
        optimizer.step()
    return loss.item()


torch.manual_seed(0)
net = make_net()
train(net, torch.optim.Adam(net.parameters(), lr=0.01), 300)

keras.utils.set_random_seed(0)
net_keras = keras.Sequential([keras.Input((4,)), keras.layers.Dense(16, activation="relu"), keras.layers.Dense(1)])
net_keras.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
history = net_keras.fit(scaled[train_rows], jan_mean[train_rows], epochs=300, batch_size=800, verbose=0)

**1.** Print the names and shapes in the state dict of `torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.ReLU(), torch.nn.Linear(8, 8), torch.nn.ReLU(), torch.nn.Linear(8, 1))`, and say why the numbers in the names skip.

In [3]:
deeper = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.ReLU(), torch.nn.Linear(8, 8), torch.nn.ReLU(),
                             torch.nn.Linear(8, 1))
for name, tensor in deeper.state_dict().items():
    print(f"{name:9}", tuple(tensor.shape))

0.weight  (8, 4)
0.bias    (8,)
2.weight  (8, 8)
2.bias    (8,)
4.weight  (1, 8)
4.bias    (1,)


Positions 1 and 3 are the two `ReLU` layers, which hold no tensors, so the names go from 0 to 2 to 4.

**2.** Save `net`'s state dict to `scratch/station_net.pt`, load the file into a new network from `make_net`, then add 1 to that network's first weight. Load the file again and show that it still holds the old value.

In [4]:
torch.save(net.state_dict(), SCRATCH / "station_net.pt")
copy_net = make_net()
copy_net.load_state_dict(torch.load(SCRATCH / "station_net.pt"))
with torch.no_grad():
    copy_net[0].weight[0, 0] += 1.0

saved = torch.load(SCRATCH / "station_net.pt")
print(round(copy_net[0].weight[0, 0].item(), 3), round(saved["0.weight"][0, 0].item(), 3),
      round(net[0].weight[0, 0].item(), 3))

1.257 0.257 0.257


The new network's weight moved from 0.257 to 1.257, and the file still holds 0.257, as `net` does. The file holds what `torch.save` wrote, and nothing has written to it since, so a change to a network loaded from it reaches neither the file nor `net`.

**3.** Train a new network from `torch.manual_seed(0)` for 100 steps, save a checkpoint to `scratch/checkpoint.pt` that also holds the last training loss, and load it and print the step and the loss.

In [5]:
torch.manual_seed(0)
model = make_net()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
last_loss = train(model, optimizer, 100)
torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(), "step": 100, "loss": last_loss},
           SCRATCH / "checkpoint.pt")

checkpoint = torch.load(SCRATCH / "checkpoint.pt")
print("step", checkpoint["step"], "| loss", round(checkpoint["loss"], 3))

step 100 | loss 1.676


The checkpoint is a dictionary, so it can hold a number beside the state dicts, and `torch.load` gives it back.

**4.** Save `net_keras` to `scratch/copy.keras`, load it, train the copy for 10 more epochs with `fit`, and print its optimizer's step count before and after.

In [6]:
net_keras.save(SCRATCH / "copy.keras")
copy_keras = keras.models.load_model(SCRATCH / "copy.keras")
print("before", int(copy_keras.optimizer.iterations.numpy()))
copy_keras.fit(scaled[train_rows], jan_mean[train_rows], epochs=10, batch_size=800, verbose=0)
print("after", int(copy_keras.optimizer.iterations.numpy()))

before 300
after 310


The loaded optimizer carried on counting from the 300 steps `net_keras` took, and 10 epochs of one full batch each added 10.

The answers wrote their files in `scratch`, which this cell removes.

In [7]:
shutil.rmtree(SCRATCH)
print("scratch removed:", not SCRATCH.exists())

scratch removed: True


---

&#8592; **Back to:** [Saving and Loading Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/14-saving-and-loading-models.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
